# Spike sorting pipeline with kilosort4 demonstration notebook

This notebook is to demonstrate how spike sorting pipeline works step-by-step.

It is only used for demonstration (because it's slow and stops on each step). If you run sorting for real experiments, it is more recommended to use run_sorting_pipeline.py with your session config.

```bash
python run_sorting_pipeline.py   --config session_demo_1probe
# ... curate in Phy ...
python run_exporting_pipeline.py --config session_demo_1probe
```

This notebook shows the by-step logic behind run_sorting_pipeline.py, up to curation in Phy; what follows curation is in step2_run_export_stepbystep.ipynb. What is run below (except for the gui) is the same as what is doing inside run_sorting_pipeline.py.

### Optional step1: Check the recording using kilosort GUI

It is recommended by the official manuel that it's better to check the data first from kilosort gui. If plan to do so, just use terminal or just run the blow cell to turn on GUI. The instruction for the gui is here : https://kilosort.readthedocs.io/en/latest/gui_guide.html

The kilosort manuel said using GUI could be helpful in finding out bad channels, or deciding whether to use whitening, etc

In [5]:
!python -m kilosort

You can run the sorting process through GUI directly actually. But run through script has more flexibility for batch sorting.

In [7]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

import spikesorting as ss

print(REPO)
print("verbs:", ", ".join(n for n in ss.__all__ if n[0].islower()))

c:\SpikeSorting\SpikeSortingKilosort
verbs: skip_reason, load_session_config, setup_probe, sort_with_kilosort, extract_sync, extract_lfp, time_remapping, validate_remapping, export_results, export_waveforms, stamp_lfp_timebase, load_machine, stream_label


## 1. Load the session configuration

In the first step before real soring begins, it set up two configurations:

 **the machine**: where this script run. For 90% of time, I bet this will be run on the windows rig/sorting machine or HPC. You don't need to modify the default settings there usually. 

 **the session**: what parameters we will use in this session.
 I have established 4 templates which should cover most of cases in the lab. 
 
 For this demo, choose session_demo_1probe.yaml. 

 Inside the session config yaml, it defines the path for the recording data (.bin) and several parameters 
 related to the sorting. 

 For your own experiment, feel free to copy one config yaml file template and make modifications based on your experiments.




In [10]:
config = ss.load_session_config(REPO / "configs" / "session_demo_1probe.yaml", "windows_rig")
try:
    config.paths.mkdirs()
except OSError as error:
    print(f'could not create output dirs: {error}\n')

print(f"session:   {config.session}")
print(f"machine:   {config.machine.name}  (device={config.machine.device})")

# cache is where the pipeline actually run on. It usually points to a high speed SSD on the local PC. 
# When sorting completes, the output will be moved to the data folder for convenience.
print(f"cache:     {config.cache_dir}")
# CatGT is a command line tool for edge extraction of the 1Hz synchronize pulse.If CatGT is not installed, I have added a python-based script for extraction.
print(f"CatGT:     {config.machine.catgt_dir or 'not installed -> NumPy fallback'}")
# Tprime is a command line tool for alignment and time remap.
# The above two tools will be in need in the exproting step after sorting.


session:   session_demo_1probe
machine:   windows_rig  (device=cuda)
cache:     D:\kilosort_cache
CatGT:     C:\SpikeGLX\CommandLineTools\CatGTWinApp\CatGT-win


## 2. Set up the probe

Kilosort relies heavily on the configuration of the probe. It needs those information to infer spikes from nearby channels. A wrong probe file will lead to a sorting failure. 

Luckily, Neuropixel data usually saves with a meta file containing it's recording probe information.

So it can be automatically retrived through the meta file. For Utah array recording, the probe file can be retrived through cmp file. 

But if a meta file or a cmp file is not founded, you have to define the dir for a probe file manuelly. Otherwise it will report error.

| | Neuropixels | Blackrock |
|---|---|---|
| 1 | the run's `.meta` (`~snsGeomMap`) | `blackrock.probe_file` |
| 2 | `neuropixels.probe_file` | **raises** -- no honest default |
| 3 | **raises** -- no honest default | |

One key per system, whatever the map's format: `probe_file` is read by its
extension -- `.cmp`, `.json` or `.mat`.

This demo session does not come with it's meta file. So it will load through the probe_file downloaded throug h the setup_demo_data notebook.


In [17]:
# The probe file is already setup throug the session_demo_1probe.yaml file.
probe_file = config.neuropixels.probe_file
print(f'The probe config: {probe_file}')
SYSTEM = "neuropixels" #it can be neuropxiels or blackrock. Once define it, the following setup probe will automatically setup probe through probe_file
probe = ss.setup_probe(config, SYSTEM)
print(f"{probe['n_chan']} channels, placeholder geometry: {probe.get('_placeholder', False)}")
print(f"  x positions : {sorted(set(np.asarray(probe['xc']).tolist()))}")
print(f"  y range     : {min(probe['yc'])} -> {max(probe['yc'])} um")
print(f"  shanks      : {len(set(np.asarray(probe['kcoords']).tolist()))}")
#If meta file is not recognized and there is no file in probe_file defined, it will report error.

The probe config: C:\SpikeSorting\SpikeSortingKilosort\configs\probes\np1_default.json
384 channels, placeholder geometry: False
  x positions : [11.0, 27.0, 43.0, 59.0]
  y range     : 20.0 -> 3840.0 um
  shanks      : 1


In [ ]:
# Plot the probe as Kilosort will see it. Left: every contact, coloured by shank.
# Right: the bottom ZOOM_UM of the probe, each contact labelled with its chanMap
# entry -- the row in the binary that contact reads. On a linear probe the numbers
# should climb steadily with depth; a jump or a reversal means a miswired map.
ZOOM_UM = 200.0

xc, yc = np.asarray(probe["xc"], float), np.asarray(probe["yc"], float)
chan_map, shank = np.asarray(probe["chanMap"]), np.asarray(probe["kcoords"])

fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(10, 7), gridspec_kw={"width_ratios": [1, 2]})

ax_full.scatter(xc, yc, s=6, marker="s", c=shank, cmap="tab10", vmin=0, vmax=9)
ax_full.set_xlim(xc.min() - 20, xc.max() + 20)
ax_full.set_xlabel("x (um)")
ax_full.set_ylabel("depth from tip (um)")
ax_full.set_title(f"{probe['n_chan']} contacts, {np.unique(shank).size} shank(s)")
ax_full.axhspan(yc.min() - 5, yc.min() + ZOOM_UM, color="tab:orange", alpha=0.15)

near_tip = yc <= yc.min() + ZOOM_UM
ax_zoom.scatter(xc[near_tip], yc[near_tip], s=200, marker="s", color="lightgrey", edgecolor="k")
for x, y, ch in zip(xc[near_tip], yc[near_tip], chan_map[near_tip]):
    ax_zoom.text(x, y, str(ch), ha="center", va="center", fontsize=7)
ax_zoom.set_aspect("equal")
ax_zoom.set_xlim(xc.min() - 15, xc.max() + 15)
ax_zoom.set_xlabel("x (um)")
ax_zoom.set_title(f"bottom {ZOOM_UM:g} um, labelled by chanMap (row in the binary)")

fig.suptitle(f"probe: {Path(probe_file).name if probe_file else 'from the .meta'}")
fig.tight_layout()
plt.show()

## 3. Preprocess: CatGT or SpikeInterface, before Kilosort

`preprocess(config, system)` writes the binary Kilosort will read: the recording
band-passed and re-referenced, as the session's `<system>.preprocess:` block says.
Kilosort then only whitens -- its own CAR (`do_CAR`) is always off.

- **Neuropixels**: `method: catgt` (the default) runs CatGT `-apfilter=butter,12,300,10000
  -gblcar -gfix` with its tshift, and extracts the sync edges in the same pass.
  `method: spikeinterface` does phase shift, band-pass and median CAR in SpikeInterface.
  CatGT that cannot run here (no `catgt_dir`, e.g. this laptop) **stops with an error**;
  it never switches to SpikeInterface by itself -- change the session if that is what you want.
- **Blackrock**: SpikeInterface only (CatGT cannot read a `.ns6`).
- `motion: dredge` (Neuropixels) corrects drift in SpikeInterface instead of Kilosort's `nblocks`, which must then be 0.

The result goes to `<system>_dir/preprocessed/` with `preprocess.json` saying what made it,
and is reused while that still matches. `sort_with_kilosort` calls this itself, so this
cell only makes the step visible.

In [ ]:
print("will it preprocess?", ss.skip_reason(config, ss.preprocess, SYSTEM) or "yes")
print("  ", config.preprocess_for(SYSTEM))
try:
    binary = ss.preprocess(config, SYSTEM)
    print(binary.summary() if binary else "nothing to preprocess")
except Exception as error:
    print(f"cannot preprocess on this machine: {type(error).__name__}: {error}")

## 4. Sort with Kilosort4

`sort_with_kilosort(config, system)` resolves the map,
takes the preprocessed binary, and calls Kilosort4's own `run_kilosort` on it.

**What it hands Kilosort** is the preprocessed binary from step 3 (made here if step 3
was skipped). Kilosort is always called with `do_CAR=False`; what it still does on every
batch -- its own highpass at `highpass_cutoff`, whitening, drift correction at `nblocks` --
is named under `kilosort:` in the session config. `config.kilosort_for(system)` is that
block, split here into Kilosort's `settings` dict and `run_kilosort`'s own arguments by
asking the installed Kilosort which is which. `bad_channels` removes a list you supply.

Sorting runs in the machine's SSD cache and the results are copied out, so nothing large
is written over the network share. Returns `None` without sorting when
`kilosort_on_<system>` is false.

This is the slow cell and the only one needing a GPU. On a real session run it from
`run_sorting_pipeline.py` so it survives a closed laptop, or send it to a cluster:

```bash
python run_sorting_pipeline.py --config <s>
```

### Below I first run a dry-run, which is not sorting anything, but to check whether the file is ready: Whether the data file can be detected, whether probe file can be deteced, etc.

In [18]:
# dry_run resolves everything and sorts nothing, so it works on a laptop.
print("will it sort?", ss.skip_reason(config, ss.sort_with_kilosort, SYSTEM) or "yes")
for key, value in (ss.sort_with_kilosort(config, SYSTEM, dry_run=True) or {}).items():
    print(f"  {key:32} {value}")

will it sort? yes
  probe                            {'n_chan': 384, 'x_range_um': [11.0, 59.0], 'y_range_um': [20.0, 3840.0], 'n_groups': 1, 'placeholder_geometry': False}
  binary                           Z:\Monkey neuropixel_demo\demo\ZFM-02370_mini.imec0.ap.short.bin (385 channels at 30000 Hz)
  results_dir                      Z:\Monkey neuropixel_demo\demo\kilosort4
  settings                         {'n_chan_bin': 385, 'fs': 30000.0}
  run_kilosort arguments           {}


### Now it's the real run on kilosort4, it may takes a while....

In [19]:
try:
    result = ss.sort_with_kilosort(config, SYSTEM)
    print(result.summary() if result else "nothing to sort")
except Exception as error:
    print(f"cannot sort on this machine: {type(error).__name__}: {error}")

kilosort.run_kilosort: Kilosort version 4.1.7
kilosort.run_kilosort: Python version 3.11.15
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: System information:
kilosort.run_kilosort: Windows-10-10.0.26200-SP0 AMD64
kilosort.run_kilosort: Intel64 Family 6 Model 151 Stepping 2, GenuineIntel
kilosort.run_kilosort: Using CUDA device: NVIDIA GeForce RTX 5060 7.96GB
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: Sorting [WindowsPath('Z:/Monkey neuropixel_demo/demo/ZFM-02370_mini.imec0.ap.short.bin')]
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage before sorting
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:     4.10 %
kilosort.run_kilosort: Mem used:     47.60 %     |      15.02 GB
kilosort.run_kilosort: Mem avail:    16.54 / 31.56 GB
kilosort.run_kilosort: ------------------------------------------------------
kilosort.run_kilosort

280 units, 431328 spikes at 30000 Hz -> Z:\Monkey neuropixel_demo\demo\kilosort4


### If sorting Utah array on Blackrock...

Nothing changes but the argument. The map comes from the array's `.cmp` rather than the
run's `.meta`, and the `.ns6` is transformed into a binary on the way in -- but that is
inside the verb, and the call is the same.

`skip_reason` is what the runner scripts print for a `[--]` line: which systems recorded
is derived from the paths the session declares, and whether to sort them is the one
thing paths cannot say.

In [ ]:
for system in ss.SYSTEMS:
    print(f"{system:12} declared={config.has_data(system)} "
          f"sorts={getattr(config, f'sorts_{system}')} "
          f"-> {config.paths.dir_for(system)}")
    for verb in (ss.setup_probe, ss.sort_with_kilosort):
        why = ss.skip_reason(config, verb, system)
        print(f"    {verb.__name__:24} {why or 'will run'}")

## 5. Curate in Phy

Now for this step, go to the kilosort4 folder, right click, open a terminal, and do manuel curation using phy.
```bash
conda activate phy
```

```bash
phy template-gui params.py
```

Phy writes `cluster_group.tsv`, whose labels override Kilosort's own
`cluster_KSLabel.tsv`. Everything downstream reads the human labels when they exist.

## Next: the exporting pipeline

Everything after curation -- extracting the sync pulses, mapping spike times onto
Blackrock time, validating that map, and exporting spike times, waveforms and quality
metrics -- is `run_exporting_pipeline.py`, walked through step by step in
**`step2_run_export_stepbystep.ipynb`**.

```bash
conda activate kilosort4      # the waveform export reads the recording through Kilosort
python run_exporting_pipeline.py --config <session>
```